In [ ]:
from collections import defaultdict, Counter
import pandas as pd
# ============================================================
# DATASET
# ============================================================
corpus = [
    [
        ("Mary", "N"),
        ("Jane", "N"),
        ("can", "M"),
        ("see", "V"),
        ("Will", "N"),
    ],
    [
        ("Spot", "N"),
        ("will", "M"),
        ("see", "V"),
        ("Mary", "N"),
    ],
    [
        ("Will", "M"),
        ("Jane", "N"),
        ("spot", "V"),
        ("Mary", "N"),
    ],
    [
        ("Mary", "N"),
        ("will", "M"),
        ("see", "V"),
        ("Spot", "N"),
    ],
]
# ============================================================
# STEP 1: Add <s> and <\s> to every sentence
# ============================================================
sentences = []
for sentence in corpus:
    new_sentence = [("<s>", "<s>")]
    new_sentence.extend(sentence)
    new_sentence.append(("<\\s>", "<\\s>"))
    sentences.append(new_sentence)
# Extract only tags
tag_sequences = [
    [tag for word, tag in sentence]
    for sentence in sentences
]
print("STEP 1: TAG SEQUENCES")
print("=" * 60)
for sequence in tag_sequences:
    print(" ".join(sequence))
# ============================================================
# STEP 2: FORM UNIQUE VALID BIGRAMS
# ============================================================
def form_bigrams(tag_sequences):
    bigrams = set()
    for sequence in tag_sequences:
        for i in range(len(sequence) - 1):
            current_tag = sequence[i]
            next_tag = sequence[i + 1]
            # Ignore <s> -> <\s>
            if current_tag == "<s>" and next_tag == "<\\s>":
                continue
            bigrams.add((current_tag, next_tag))
    return sorted(bigrams)
bigrams = form_bigrams(tag_sequences)
print("\nSTEP 2: UNIQUE VALID BIGRAMS")
print("=" * 60)
for bigram in bigrams:
    print(bigram)
# ============================================================
# STEP 3: TRANSITION PROBABILITIES
# ============================================================
def transition_probabilities(tag_sequences):
    transitions = defaultdict(Counter)
    for sequence in tag_sequences:
        for i in range(len(sequence) - 1):
            current_tag = sequence[i]
            next_tag = sequence[i + 1]
            # Ignore <s> -> <\s>
            if current_tag == "<s>" and next_tag == "<\\s>":
                continue
            transitions[current_tag][next_tag] += 1
    transition_matrix = {}
    for current_tag, next_tags in transitions.items():
        total = sum(next_tags.values())
        transition_matrix[current_tag] = {
            next_tag: f"{count}/{total}"
            for next_tag, count in next_tags.items()
        }
    return transition_matrix
trans_matrix = transition_probabilities(tag_sequences)
print("\nSTEP 3: CONDITIONAL PROBABILITIES")
print("=" * 60)
for current_tag, probabilities in trans_matrix.items():
    for next_tag, probability in probabilities.items():
        print(
            f"P({next_tag} | {current_tag}) = {probability}"
        )
# ============================================================
# STEP 4: TRANSITION PROBABILITY MATRIX
# ============================================================
tag_order = ["<s>", "N", "M", "V", "<\\s>"]
df_transition = pd.DataFrame(
    "0",
    index=tag_order,
    columns=tag_order
)
for current_tag, next_tags in trans_matrix.items():
    for next_tag, fraction in next_tags.items():
        if current_tag in df_transition.index and \
           next_tag in df_transition.columns:
            df_transition.loc[current_tag, next_tag] = fraction
print("\nSTEP 4: TRANSITION PROBABILITY MATRIX")
print("=" * 60)
display(df_transition)
# ============================================================
# STEP 5: EMISSION PROBABILITIES
# ============================================================
def emission_probabilities(sentences):
    emissions = defaultdict(Counter)
    for sentence in sentences:
        for word, tag in sentence:
            # Ignore sentence boundaries
            if word in ("<s>", "<\\s>"):
                continue
            # Convert words to lowercase
            word = word.lower()
            emissions[word][tag] += 1
    emission_matrix = {}
    for word, tags in emissions.items():
        total = sum(tags.values())
        emission_matrix[word] = {
            tag: f"{count}/{total}"
            for tag, count in tags.items()
        }
    return emission_matrix
emis_matrix = emission_probabilities(sentences)
# Unique words
unique_words = sorted(emis_matrix.keys())
# POS tags
pos_tags = ["N", "M", "V"]
df_emission = pd.DataFrame(
    "0",
    index=unique_words,
    columns=pos_tags
)
for word, tags in emis_matrix.items():
    for tag, fraction in tags.items():
        if tag in df_emission.columns:
            df_emission.loc[word, tag] = fraction
print("\nSTEP 5: EMISSION PROBABILITY MATRIX")
print("=" * 60)
display(df_emission)

STEP 1: TAG SEQUENCES
<s> N N M V N <\s>
<s> N M V N <\s>
<s> M N V N <\s>
<s> N M V N <\s>

STEP 2: UNIQUE VALID BIGRAMS
('<s>', 'M')
('<s>', 'N')
('M', 'N')
('M', 'V')
('N', '<\\s>')
('N', 'M')
('N', 'N')
('N', 'V')
('V', 'N')

STEP 3: CONDITIONAL PROBABILITIES
P(N | <s>) = 3/4
P(M | <s>) = 1/4
P(N | N) = 1/9
P(M | N) = 3/9
P(<\s> | N) = 4/9
P(V | N) = 1/9
P(V | M) = 3/4
P(N | M) = 1/4
P(N | V) = 4/4

STEP 4: TRANSITION PROBABILITY MATRIX


,<s>,N,M,V,<\s>
<s>,0,3/4,1/4,0,0
N,0,1/9,3/9,1/9,4/9
M,0,1/4,0,3/4,0
V,0,4/4,0,0,0
<\s>,0,0,0,0,0



STEP 5: EMISSION PROBABILITY MATRIX


,N,M,V
can,0,1/1,0
jane,2/2,0,0
mary,4/4,0,0
see,0,0,3/3
spot,2/3,0,1/3
will,1/4,3/4,0


In [ ]:
import numpy as np
import pandas as pd

# Helper function to convert fraction strings to floats
def fraction_to_float(fraction_str):
    if isinstance(fraction_str, (int, float)):
        return float(fraction_str)
    if '/' in fraction_str:
        numerator, denominator = map(int, fraction_str.split('/'))
        return numerator / denominator
    return float(fraction_str)

# Convert DataFrames to float probabilities
df_transition_float = df_transition.map(fraction_to_float)
df_emission_float = df_emission.map(fraction_to_float)

# Ensure all relevant tags are present in both matrices (excluding boundary tags)
all_tags = sorted(list(set(tag_order) - {'<s>', '<\\s>'}))

def viterbi_algorithm(sentence, transition_matrix, emission_matrix, all_tags, start_tag='<s>', end_tag='<\\s>'):
    sentence_lower = [word.lower() for word in sentence]
    N = len(sentence_lower)

    print("\n=== Viterbi Algorithm Steps ===")
    print(f"Sentence: {sentence}")

    # Initialize Viterbi and Backpointer tables
    viterbi = pd.DataFrame(0.0, index=sentence_lower, columns=all_tags)
    backpointer = pd.DataFrame(None, index=sentence_lower, columns=all_tags, dtype=object)

    # --- Initialization Step ---
    first_word = sentence_lower[0]
    print("\n--- Initialization (First Word) ---")
    for tag in all_tags:
        trans_prob = transition_matrix.loc[start_tag, tag] if tag in transition_matrix.columns else 0.0
        emis_prob = emission_matrix.loc[first_word, tag] if first_word in emission_matrix.index and tag in emission_matrix.columns else 1e-10

        viterbi.loc[first_word, tag] = trans_prob * emis_prob
        print(f"  V[{first_word}][{tag}] = P({tag}|<s>) * P({first_word}|{tag}) = {trans_prob:.4f} * {emis_prob:.4f} = {viterbi.loc[first_word, tag]:.8f}")

    # --- Recursion Step ---
    for i in range(1, N):
        current_word = sentence_lower[i]
        prev_word = sentence_lower[i-1]
        print(f"\n--- Recursion Step for '{sentence[i]}' ---")

        for current_tag in all_tags:
            max_prob = -1.0
            best_prev_tag = None
            emis_prob = emission_matrix.loc[current_word, current_tag] if current_word in emission_matrix.index and current_tag in emission_matrix.columns else 1e-10

            for prev_tag in all_tags:
                trans_prob = transition_matrix.loc[prev_tag, current_tag] if current_tag in transition_matrix.columns else 0.0
                path_prob = viterbi.loc[prev_word, prev_tag] * trans_prob * emis_prob

                if path_prob > max_prob:
                    max_prob = path_prob
                    best_prev_tag = prev_tag

            viterbi.loc[current_word, current_tag] = max_prob
            backpointer.loc[current_word, current_tag] = best_prev_tag
            print(f"  V[{current_word}][{current_tag}] = Max path prob = {max_prob:.8f} (from prev tag '{best_prev_tag}')")

    # --- Termination Step ---
    print("\n--- Termination (Ending at <\\s>) ---")
    max_prob_end = -1.0
    last_tag = None
    last_word = sentence_lower[-1]

    for tag in all_tags:
        trans_prob_to_end = transition_matrix.loc[tag, end_tag] if end_tag in transition_matrix.columns else 0.0
        path_prob_end = viterbi.loc[last_word, tag] * trans_prob_to_end
        print(f"  From tag '{tag}': V[{last_word}][{tag}] * P(<\\s>|{tag}) = {viterbi.loc[last_word, tag]:.8f} * {trans_prob_to_end:.4f} = {path_prob_end:.8f}")

        if path_prob_end > max_prob_end:
            max_prob_end = path_prob_end
            last_tag = tag

    print(f"Best final tag: '{last_tag}' with probability {max_prob_end:.8f}")

    # --- Backtracking Step ---
    print("\n--- Path Backtracking ---")
    best_path = [last_tag]
    for i in range(N - 1, 0, -1):
        curr_word = sentence_lower[i]
        curr_tag = best_path[0]
        prev_tag = backpointer.loc[curr_word, curr_tag]
        best_path.insert(0, prev_tag)
        print(f"  At word '{sentence[i]}', Tag: '{curr_tag}' -> backpointer points to '{prev_tag}'")

    return list(zip(sentence, best_path))

# Test sentence
test_sentence = ['Mary', 'will', 'see', 'Jane']
tagged_sentence = viterbi_algorithm(test_sentence, df_transition_float, df_emission_float, all_tags)
print(f"\nFinal Output: {tagged_sentence}")


=== Viterbi Algorithm Steps ===
Sentence: ['Mary', 'will', 'see', 'Jane']

--- Initialization (First Word) ---
  V[mary][M] = P(M|<s>) * P(mary|M) = 0.2500 * 0.0000 = 0.00000000
  V[mary][N] = P(N|<s>) * P(mary|N) = 0.7500 * 1.0000 = 0.75000000
  V[mary][V] = P(V|<s>) * P(mary|V) = 0.0000 * 0.0000 = 0.00000000

--- Recursion Step for 'will' ---
  V[will][M] = Max path prob = 0.18750000 (from prev tag 'N')
  V[will][N] = Max path prob = 0.02083333 (from prev tag 'N')
  V[will][V] = Max path prob = 0.00000000 (from prev tag 'M')

--- Recursion Step for 'see' ---
  V[see][M] = Max path prob = 0.00000000 (from prev tag 'M')
  V[see][N] = Max path prob = 0.00000000 (from prev tag 'M')
  V[see][V] = Max path prob = 0.14062500 (from prev tag 'M')

--- Recursion Step for 'Jane' ---
  V[jane][M] = Max path prob = 0.00000000 (from prev tag 'M')
  V[jane][N] = Max path prob = 0.14062500 (from prev tag 'V')
  V[jane][V] = Max path prob = 0.00000000 (from prev tag 'M')

--- Termination (Ending at 

In [ ]:
def get_viterbi_tables(sentence, transition_matrix, emission_matrix, all_tags, start_tag='<s>', end_tag='<\\s>'):
    sentence_lower = [word.lower() for word in sentence]
    N = len(sentence_lower)

    viterbi = pd.DataFrame(0.0, index=sentence_lower, columns=all_tags)
    backpointer = pd.DataFrame(None, index=sentence_lower, columns=all_tags, dtype=object)

    # Initialization
    first_word = sentence_lower[0]
    for tag in all_tags:
        trans_prob = transition_matrix.loc[start_tag, tag] if tag in transition_matrix.columns else 0.0
        emis_prob = emission_matrix.loc[first_word, tag] if first_word in emission_matrix.index and tag in emission_matrix.columns else 1e-10
        viterbi.loc[first_word, tag] = trans_prob * emis_prob

    # Recursion
    for i in range(1, N):
        current_word = sentence_lower[i]
        prev_word = sentence_lower[i-1]
        for current_tag in all_tags:
            max_prob = -1.0
            best_prev_tag = None
            emis_prob = emission_matrix.loc[current_word, current_tag] if current_word in emission_matrix.index and current_tag in emission_matrix.columns else 1e-10
            for prev_tag in all_tags:
                trans_prob = transition_matrix.loc[prev_tag, current_tag] if current_tag in transition_matrix.columns else 0.0
                path_prob = viterbi.loc[prev_word, prev_tag] * trans_prob * emis_prob
                if path_prob > max_prob:
                    max_prob = path_prob
                    best_prev_tag = prev_tag
            viterbi.loc[current_word, current_tag] = max_prob
            backpointer.loc[current_word, current_tag] = best_prev_tag

    return viterbi, backpointer

v_df, bp_df = get_viterbi_tables(test_sentence, df_transition_float, df_emission_float, all_tags)

print("=== Viterbi Trellis Probability Table ===")
display(v_df)

print("\n=== Backpointer Table ===")
display(bp_df)

=== Viterbi Trellis Probability Table ===


,M,N,V
mary,0.0000,0.750000,0.000000
will,0.1875,0.020833,0.000000
see,0.0000,0.000000,0.140625
jane,0.0000,0.140625,0.000000



=== Backpointer Table ===


,M,N,V
mary,NaN,NaN,NaN
will,N,N,M
see,M,M,M
jane,M,V,M
